# Lab 10 — Mini Project: School Management System

In [1]:
# This lab uses only Python's built-in features, but this cell keeps the
# notebook runnable from a fresh kernel. It installs nothing extra.
!pip install --upgrade pip


## Step 1 — Person base class

In [2]:
# A base class captures what every member has in common: a name and an age.
class Person:
    # --- constructor: store the two shared attributes ---
    def __init__(self, name, age):
        self.name = name  # instance attribute, unique per object
        self.age = age

    # --- role_info: placeholder that subclasses override (polymorphism) ---
    def role_info(self):
        return f"Base person {self.name}, age {self.age}"

    # --- __str__: friendly text shown by print()/str() ---
    def __str__(self):
        return f"{self.name} (age {self.age})"

    # --- __repr__: unambiguous, debug-friendly representation ---
    def __repr__(self):
        return f"Person('{self.name}', {self.age})"


## Step 2 — Student subclass (encapsulation)

In [3]:
# Student inherits from Person: it reuses name/age and adds its own fields.
class Student(Person):
    # --- constructor: set up shared state, then Student-specific fields ---
    def __init__(self, name, age, course, marks):
        super().__init__(name, age)  # let Person set up name and age
        self.course = course
        self._marks = marks  # single underscore -> 'private' by convention

    # --- property: expose the private _marks as a read-able attribute ---
    @property
    def marks(self):
        return self._marks

    # --- property setter: validate a value before storing it ---
    @marks.setter
    def marks(self, value):
        if value < 0 or value > 100:
            raise ValueError("Marks must be between 0 and 100")
        self._marks = value

    # --- override role_info to return Student-specific info ---
    def role_info(self):
        return f"Student {self.name} takes {self.course} and scored {self.marks}"

    # --- __str__: override the friendly display for a student ---
    def __str__(self):
        return f"{self.name} ({self.course})"


## Step 3 — Teacher subclass (polymorphism)

In [4]:
# Teacher also inherits from Person and overrides role_info.
class Teacher(Person):
    # --- constructor: reuse the base constructor, then add the subject ---
    def __init__(self, name, age, subject):
        super().__init__(name, age)  # reuse the base constructor
        self.subject = subject

    # --- override role_info to return Teacher-specific info ---
    def role_info(self):
        return f"Teacher {self.name} teaches {self.subject}"


## Step 4 — School with inner Classroom class

In [5]:
# School is the master class: it owns inner Classroom objects and counts
# every member added across all of them.
class School:
    # --- class attribute: shared counter, updated via a classmethod ---
    total_members = 0  # shared by every School object

    # --- constructor ---
    def __init__(self, name):
        self.name = name
        # each School holds a list of its Classrooms
        self.classrooms = []

    # --- classmethod: keep the shared member counter in sync ---
    # A classmethod receives the class, so it can read/write cls.total_members.
    @classmethod
    def increment_members(cls):
        cls.total_members += 1

    # --- nested Classroom class: belongs to exactly one School ---
    class Classroom:
        def __init__(self, parent, subject):
            self.parent = parent  # back-reference to the owning School
            self.subject = subject
            self.roster = []  # people assigned to this classroom

        def add_member(self, person):
            # Add one person to the roster, then update the global tally.
            self.roster.append(person)
            School.increment_members()

        # len(classroom) = number of people in its roster.
        def __len__(self):
            return len(self.roster)

        # Friendly one-line summary for print(classroom).
        def __str__(self):
            return f"{self.subject} ({len(self.roster)} members)"

    # --- helpers: create classrooms and route members into them ---
    def add_classroom(self, subject):
        # Build a new inner Classroom that points back at this School.
        room = self.Classroom(self, subject)
        self.classrooms.append(room)
        return room

    def add_member(self, person, classroom):
        # Delegate to the given classroom's own add_member().
        classroom.add_member(person)

    # --- dunder methods: make School feel like a natural Python object ---
    # len(school) = total people across all classrooms.
    def __len__(self):
        return sum(len(c) for c in self.classrooms)

    # A readable multi-line summary of the whole school.
    def __str__(self):
        lines = [f"{self.name} School ({len(self)} members)"]
        for c in self.classrooms:
            names = ", ".join(str(p) for p in c.roster)
            lines.append(f"  {c.subject}: {names}")
        return "\n".join(lines)


## Step 5 — Polymorphism in action

In [6]:
# Build a mixed list of students and teachers, then call the
# overridden role_info() on each one -- no type checks needed.
people = [
    Student("Ana", 15, "Math", 88),
    Student("Leo", 16, "Physics", 92),
    Teacher("Mr. Bell", 38, "Math"),
    Teacher("Ms. Ruiz", 45, "Physics"),
]

# Each object runs its own role_info() version via polymorphism.
for p in people:
    print(p.role_info())  # the object knows how to describe itself


Student Ana takes Math and scored 88
Student Leo takes Physics and scored 92
Teacher Mr. Bell teaches Math
Teacher Ms. Ruiz teaches Physics


## Step 6 — Dunder methods and property validation

In [7]:
# Demonstrate properties, validation, and magic methods on a student.
s = Student("Ana", 15, "Math", 88)

# repr/str come from the inherited and overridden dunders.
print("repr:", repr(s))
print("str:", str(s))

# marks is a property exposing the private _marks.
print("Marks property:", s.marks)

# The setter rejects an out-of-range value before it is stored.
try:
    s.marks = -5
except ValueError as e:
    print("Invalid marks rejected:", e)  # setter blocked the bad value

# Build a school and two classrooms to check the aggregation.
school = School("Sunrise High")
math = school.add_classroom("Math")
physics = school.add_classroom("Physics")

# Both totals are 0 before any member has been added.
print("len(school):", len(school))  # empty so far
print("School counter:", School.total_members)  # shared class attribute


repr: Person('Ana', 15)
str: Ana (Math)
Marks property: 88
Invalid marks rejected: Marks must be between 0 and 100
len(school): 0
School counter: 0


## Step 7 — Full system run

In [8]:
# Populate each classroom; every add bumps the shared member counter.
math.add_member(Student("Ana", 15, "Math", 88))
math.add_member(Student("Ben", 14, "Math", 72))
math.add_member(Teacher("Mr. Bell", 38, "Math"))

physics.add_member(Student("Leo", 16, "Physics", 92))
physics.add_member(Student("Mia", 15, "Physics", 81))
physics.add_member(Teacher("Ms. Ruiz", 45, "Physics"))

# print(school) triggers School.__str__, using each person's own __str__.
print(school)

# The two totals agree because both count every member added.
print("Total members (len):", len(school))  # uses School.__len__
print("Classmethod counter:", School.total_members)  # matches len


Sunrise High School (6 members)
  Math: Ana (Math), Ben (Math), Mr. Bell (age 38)
  Physics: Leo (Physics), Mia (Physics), Ms. Ruiz (age 45)
Total members (len): 6
Classmethod counter: 6
